In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib
import re

In [2]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [3]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [ ]:
louisiana

In [ ]:
[c for c in louisiana.columns if c.startswith ("frac_trns_fuelmix_road_")]

In [4]:
# 1) Filter your base-case scenario
base_case = louisiana[louisiana["primary_id"] == 0].copy()

In [14]:


# ——————————————————————————————————————————
# 2) Grab all rail electricity consumption (PJ) columns
# ——————————————————————————————————————————
rail_patterns = [r"^energy_consumption_trns_rail_.*_electricity$"]
rail_elec_cols = [
    c for c in base_case.columns
    if any(re.match(p, c) for p in rail_patterns)
]
if not rail_elec_cols:
    raise KeyError("No rail electricity consumption columns found")
# sum across any sub‐modes (freight, passenger, etc.)
rail_elec_PJ = base_case[rail_elec_cols].sum(axis=1)

# ——————————————————————————————————————————
# 3) Compute “switched to electricity” relative to baseline
# ——————————————————————————————————————————
baseline_elec = rail_elec_PJ.iloc[0]
switched_to_rail_elec_PJ = rail_elec_PJ - baseline_elec

# ——————————————————————————————————————————
# 4) Apply cost & saving multipliers ($ per PJ)
# ——————————————————————————————————————————
cost_mult   = 422_400_000   # $ per PJ
saving_mult =   2_377_710   # $ per PJ

cost_series   = switched_to_rail_elec_PJ * cost_mult
saving_series = switched_to_rail_elec_PJ * saving_mult
net_series    = cost_series - saving_series  # net capex

# ——————————————————————————————————————————
# 5) Build the output DataFrame
# ——————————————————————————————————————————
output_rail = pd.DataFrame({
    "primary_id":                          base_case["primary_id"],
    "region":                              base_case["region"],
    "time_period":                         base_case["time_period"],
    "rail_elec_consumption_PJ":            rail_elec_PJ,
    "switched_to_rail_elec_PJ":            switched_to_rail_elec_PJ,
    "rail_fuel_switch_cost_$":             cost_series,
    "rail_fuel_switch_saving_$":           saving_series,
    "rail_fuel_switch_net_cost_$":         net_series,
}, index=base_case.index)






In [18]:
output_rail

,primary_id,region,time_period,rail_elec_consumption_PJ,switched_to_rail_elec_PJ,rail_fuel_switch_cost_$,rail_fuel_switch_saving_$,rail_fuel_switch_net_cost_$
0,0,louisiana,0,0.189831,0.000000,0.000000e+00,0.000000,0.000000e+00
1,0,louisiana,1,0.176563,-0.013268,-5.604362e+06,-31547.225642,-5.572815e+06
2,0,louisiana,2,0.171586,-0.018246,-7.707021e+06,-43383.192585,-7.663638e+06
3,0,louisiana,3,0.175757,-0.014074,-5.944909e+06,-33464.180621,-5.911445e+06
4,0,louisiana,4,0.181465,-0.008366,-3.533837e+06,-19892.140609,-3.513945e+06
5,0,louisiana,5,0.128214,-0.061618,-2.602736e+07,-146509.273504,-2.588085e+07
6,0,louisiana,6,0.090183,-0.099648,-4.209127e+07,-236933.813064,-4.185434e+07
7,0,louisiana,7,0.092627,-0.097204,-4.105916e+07,-231123.992809,-4.082804e+07
8,0,louisiana,8,0.095382,-0.094450,-3.989549e+07,-224573.637153,-3.967092e+07
9,0,louisiana,9,0.098451,-0.091381,-3.859917e+07,-217276.578315,-3.838189e+07


In [19]:
output_rail = (
    output_rail
        .drop(
            columns=[
                'rail_fuel_switch_cost_$',
                'rail_fuel_switch_saving_$'
            ]
        )
        .rename(columns={
            'rail_fuel_switch_net_cost_$': 'capex'
        })
)

In [20]:
# ——————————————————————————————————————————
# 6) (Optional) save to CSV
# ——————————————————————————————————————————
OUTPUT_DIR = DATA_DIR / "output"
output_rail.to_csv(OUTPUT_DIR / "transportation_rail_fuel_switch_cost.csv", index=False)